# Building the opportunities table

## Setup

In [ ]:
import numpy as np
import pandas as pd

MIN_RUNS = 10

## Find the opportunities

In [ ]:
games = pd.read_csv("../statcast_game_data.csv")
games["season"] = games.game_date.str[:4].astype(int)

plays = games[(games.events == "single") & games.hit_location.isin([7, 8, 9]) & games.on_2b.notna()]
plays["on_2b"] = plays.on_2b.astype(int)
plays["bat_team"] = plays.home_team.where(plays.inning_topbot == "Bot", plays.away_team)

## Was the runner sent, and was he safe?

There's no way to know if the coach actually sent the runner or if the runner made his own decision. I was able to compute if the runner attempted to score though.

If there was a runner on third, it is assumed he would score on a single to the outfield. In that case, I am checking to see if a second run scored or if that runner stops at third.

With no runner on third, I am simply checking to see if the runner went home or stopped at third.

In [ ]:
runs = plays.post_bat_score - plays.bat_score
runs_needed = plays.on_3b.notna().map({True: 2, False: 1})
out_at_home = plays.des.str.contains("out at home", na=False)

plays["outcome"] = "stopped"
plays.loc[runs >= runs_needed, "outcome"] = "scored"
plays.loc[(runs == runs_needed - 1) & out_at_home, "outcome"] = "out"

plays["sent"] = plays.outcome != "stopped"
plays["safe"] = (plays.outcome == "scored").astype(float).where(plays.sent)

## Which direction was the ball hit?

I used the hit coordinates that statcast tracks in combination with the coordinates for home plate that I found to compute the spray angle.

Note: Found the formula on the internet via Gemini and double checked the values myself

In [ ]:
angle = np.degrees(np.arctan2(plays.hc_x - 125.42, 198.27 - plays.hc_y))
plays["spray_angle"] = angle.where(angle.abs() <= 60).clip(-45, 45)

## How fast is the runner?

In [ ]:
speed = pd.read_csv("../sprint_speed.csv")
speed = speed[speed.competitive_runs >= MIN_RUNS]
speed = speed.rename(columns={"player_id": "on_2b", "sprint_speed": "runner_speed"})
plays = plays.merge(speed[["on_2b", "season", "runner_speed"]], on=["on_2b", "season"], how="left")

plays["speed_filled_in"] = plays.runner_speed.isna()
median_speed = speed.groupby("season").runner_speed.median()
plays["runner_speed"] = plays.runner_speed.fillna(plays.season.map(median_speed))

## How strong is the outfielder's arm?

In [ ]:
plays["fielder"] = plays.fielder_8
plays.loc[plays.hit_location == 7, "fielder"] = plays.fielder_7
plays.loc[plays.hit_location == 9, "fielder"] = plays.fielder_9

arm = pd.read_csv("../arm_strength.csv")
arm = arm.rename(columns={"player_id": "fielder", "arm_of": "fielder_arm"})
plays = plays.merge(arm[["fielder", "season", "fielder_arm"]], on=["fielder", "season"], how="left")

plays["arm_filled_in"] = plays.fielder_arm.isna()
median_arm = arm.groupby("season").fielder_arm.median()
plays["fielder_arm"] = plays.fielder_arm.fillna(plays.season.map(median_arm))

## Save to CSV

In [ ]:
plays["complete"] = plays[["launch_speed", "launch_angle", "bb_type", "spray_angle"]].notna().all(axis=1)

columns = ["game_pk", "game_date", "season", "inning", "at_bat_number", "bat_team", "outs_when_up",
           "bb_type", "launch_speed", "launch_angle", "spray_angle", "runner_speed", "speed_filled_in",
           "fielder_arm", "arm_filled_in", "complete", "outcome", "sent", "safe", "des"]
plays[columns].to_csv("../opportunities.csv", index=False)

pd.crosstab(plays.season, plays.outcome)